# 10 — El pasaporte de cumplimiento como base de datos documental

**TFM: Predicción de emisiones de CO₂ de buques (THETIS-MRV)**

*El notebook 09 lleva el registro a SQLite
porque el dato de origen es tabular y relacional: buques y ejercicios anuales, dos tablas y una clave
ajena. Este notebook no repite ese ejercicio con otra tecnología. Plantea la pregunta al revés:*

> **el registro es relacional, pero el producto que este trabajo entrega no lo es.**

Lo que la API sirve y el panel enseña no es una fila: es el **pasaporte de cumplimiento** de un buque
—su banda CII de hoy, su trayectoria año a año hasta 2030, su factura de derechos, su balance FuelEU
y su ahorro alcanzable—. En el modelo relacional eso son cuatro tablas y tres `JOIN` para pintar una
sola ficha. En un almacén documental es **un documento y una lectura**.

Este notebook construye ese modelo documental, lo consulta con la API de MongoDB y hace las tres
comprobaciones que el proyecto exige a cualquier tecnología nueva:

1. **¿Da lo mismo?** La misma pregunta de negocio, resuelta en pandas y en una tubería de agregación,
   tiene que dar el mismo número hasta el último decimal.
2. **¿Cuánto cuesta?** Se mide, no se supone. Y si pierde, se dice.
3. **¿Encuentra algo que el otro modelo no encontraba?** Si no, la tecnología sobra.

*Nota de ejecución: se usa `mongomock`, que implementa la API de `pymongo` en memoria. El código de
consulta es el mismo que contra un servidor MongoDB real; al final se exporta la colección en JSONL
para `mongoimport`, que es la vía de despliegue.*

In [1]:
import json
import os
import time

import numpy as np
import pandas as pd
import mongomock

pd.set_option('display.width', 170)
pd.set_option('display.max_columns', 30)

RUTA_FICHA = '../reports/18_pasaporte_buques.csv'
RUTA_JSONL = '../data/processed/pasaportes.jsonl'
ANIOS = list(range(2025, 2031))

ficha = pd.read_csv(RUTA_FICHA)
print(f'{len(ficha):,} pasaportes, {ficha.shape[1]} columnas'.replace(',', '.'))
ficha[['ship_name', 'company_name', 'banda_2025', 'anio_caida', 'ets_2026_eur']].head()

14.865 pasaportes. 42 columnas


,ship_name,company_name,banda_2025,anio_caida,ets_2026_eur
0,MSC SERENA,MSC SHIPMANAGEMENT LIMITED,NaN,NaN,232485.800851
1,AQUADONNA,BLUEOCEAN SHIPMANAGEMENT INTERNATIONAL INC.,A,NaN,248026.950985
2,JEFFREYS BAY,SYNERGY PACIFIC PRIVATE LIMITED,NaN,NaN,73042.927675
3,NORD MATE,SYNERGY PACIFIC PRIVATE LIMITED,NaN,NaN,184191.595668
4,UM EDO,SYNERGY PACIFIC PRIVATE LIMITED,NaN,NaN,13495.343692


## 1. Del renglón al documento

La ficha llega como una tabla ancha: cuarenta y dos columnas por buque, seis de ellas
(`banda_2025` … `banda_2030`) que son **la misma variable repetida seis veces**. Eso es una tabla
pivotada, no un modelo de datos: si mañana el calendario de endurecimiento del CII llega a 2035
hay que añadir cinco columnas y tocar todo lo que lee la tabla.

El documento agrupa por significado, no por conveniencia de almacenamiento: una sección de identidad,
una de operación, una regulatoria con la trayectoria como **lista**, una económica. Es el mismo objeto
que la API ya devuelve en `/buques/{imo}/pasaporte`, así que el almacén documental no es una capa
nueva: **es el formato que el producto ya usa**.

In [2]:
def a_documento(f) -> dict:
    """Un renglón de la ficha -> un documento anidado. `_id` es el IMO: identidad natural."""
    def num(x):
        return None if pd.isna(x) else float(x)

    bandas = {a: getattr(f, f'banda_{a}') for a in ANIOS}
    trayectoria = [{'anio': a, 'banda': b} for a, b in bandas.items() if isinstance(b, str)]
    return {
        '_id': str(f.ship_imo_number),
        'buque': {'nombre': f.ship_name, 'tipo': f.ship_type_agrupado,
                  'capacidad_t': num(f.capacidad_estimada)},
        'naviera': {'nombre': f.company_name,
                    'imo': None if pd.isna(f.company_imo_number) else str(int(f.company_imo_number))},
        'operacion': {'anio': int(f.reporting_year), 'co2_t': num(f.total_co2_emissions_m_tonnes),
                      'horas_mar': num(f.time_spent_at_sea_hours),
                      'distancia_nm': num(f.distancia_nm), 'velocidad_nudos': num(f.velocidad_nudos)},
        'cii': {'tipo': None if pd.isna(f.cii_tipo) else f.cii_tipo,
                'obtenido': num(f.cii_attained), 'requerido': num(f.cii_requerido),
                'banda_actual': bandas[2025] if isinstance(bandas[2025], str) else None,
                'anio_caida': None if pd.isna(f.anio_caida) else int(f.anio_caida),
                'trayectoria': trayectoria},
        'ets': {'factura_2026_eur': num(f.ets_2026_eur), 'fraccion_en_ambito': num(f.factor_ambito)},
        'fueleu': {'intensidad_wtw': num(f.ghgie_wtw), 'objetivo': num(f.objetivo_fueleu),
                   'cumple': None if pd.isna(f.cumple_fueleu) else bool(f.cumple_fueleu),
                   'penalizacion_eur': num(f.penalizacion_fueleu_eur)},
        'ahorro': {'a_actividad_constante_t': num(f.ahorro_actividad_constante_t),
                   'valor_eur': num(f.ahorro_eur), 'grupo_de_comparacion': f.grupo},
    }


documentos = [a_documento(f) for f in ficha.itertuples(index=False)]
print(json.dumps(documentos[0], indent=2, ensure_ascii=False)[:1100])

{
  "_id": "1013169",
  "buque": {
    "nombre": "MSC SERENA",
    "tipo": "Container ship",
    "capacidad_t": 49207.87746170678
  },
  "naviera": {
    "nombre": "MSC SHIPMANAGEMENT LIMITED",
    "imo": "1535947"
  },
  "operacion": {
    "anio": 2025,
    "co2_t": 4524.90112,
    "horas_mar": 477.5,
    "distancia_nm": 6690.012451085024,
    "velocidad_nudos": 14.010497279759212
  },
  "cii": {
    "tipo": null,
    "obtenido": null,
    "requerido": null,
    "banda_actual": null,
    "anio_caida": null,
    "trayectoria": []
  },
  "ets": {
    "factura_2026_eur": 232485.80085122623,
    "fraccion_en_ambito": 0.604786886481179
  },
  "fueleu": {
    "intensidad_wtw": 89.12022978383833,
    "objetivo": 89.3368,
    "cumple": true,
    "penalizacion_eur": 0.0
  },
  "ahorro": {
    "a_actividad_constante_t": 844.1483888652109,
    "valor_eur": 176466.44638844437,
    "grupo_de_comparacion": "Container ship|Q2"
  }
}


**Un documento, todo el pasaporte.** Nótese lo que ha desaparecido: las seis columnas `banda_*` son
ahora una lista de longitud variable. Si el calendario del CII se amplía, el documento crece y ninguna
consulta se rompe: eso es exactamente lo que un esquema flexible compra, y es la primera vez en el
trabajo que hace falta.

In [3]:
cliente = mongomock.MongoClient()
coleccion = cliente['tfm_mrv']['pasaportes']
t0 = time.time()
coleccion.insert_many(documentos)
t_carga = time.time() - t0

# Indices: los mismos que se crearian en produccion, y por los mismos motivos.
coleccion.create_index('naviera.imo')
coleccion.create_index('cii.banda_actual')
coleccion.create_index('cii.anio_caida')
print(f'{coleccion.count_documents({}):,} documentos en {t_carga:.2f} s'.replace(',', '.'))

14.865 documentos en 0.85 s


## 2. Consulta 1: la ficha de un buque, que es donde el modelo documental gana

Esta es la consulta que sirve el panel y la que más veces se ejecuta en producción. En el modelo
relacional del notebook 09 pediría un `JOIN` de la dimensión con la tabla de hechos, más las tablas
regulatoria y económica; aquí es una lectura por clave primaria.

In [4]:
# El caso demostrativo del simulador, el CRUISE BARCELONA, es un ro-pax: el CII de MARPOL no lo
# califica, asi que su documento no trae banda. Eso es informacion, no un hueco, y la seccion 6 lo
# usa. Para ensenar la ficha completa se toma un buque calificado con fecha de caida.
caso = coleccion.find_one({'cii.banda_actual': {'$ne': None}, 'cii.anio_caida': {'$ne': None}})
print(caso['buque']['nombre'], '·', caso['buque']['tipo'], '·', caso['naviera']['nombre'])
print('banda CII hoy:', caso['cii']['banda_actual'], '| pierde la C en:', caso['cii']['anio_caida'])
print('factura ETS 2026: {:,.0f} EUR'.format(caso['ets']['factura_2026_eur']).replace(',', '.'))
print('trayectoria:', ' '.join(f"{t['anio']}:{t['banda']}" for t in caso['cii']['trayectoria']))

BIRD OF PARADISE · Bulk carrier · STELLAR SHIPMANAGEMENT INC.
banda CII hoy: B | pierde la C en: 2030
factura ETS 2026: 68.809 EUR
trayectoria: 2025:B 2026:C 2027:C 2028:C 2029:C 2030:D


## 3. Consulta 2: la pregunta de negocio, en una tubería de agregación

*«¿Qué navieras concentran más buques que pierden la C antes de 2028, y cuánto pagan de derechos?»*

Es la consulta que convierte el trabajo en un aviso temprano, y es de las que el modelo documental
hace bien: filtra por un campo anidado, agrupa por otro y ordena. Se escribe con `$match`, `$group`
y `$sort`, que es el equivalente documental de `WHERE`, `GROUP BY` y `ORDER BY`.

In [5]:
TUBERIA = [
    {'$match': {'cii.anio_caida': {'$ne': None, '$lte': 2027}}},
    {'$group': {'_id': '$naviera.nombre',
                'buques_en_riesgo': {'$sum': 1},
                'factura_ets_eur': {'$sum': '$ets.factura_2026_eur'},
                'ahorro_alcanzable_eur': {'$sum': '$ahorro.valor_eur'}}},
    {'$sort': {'buques_en_riesgo': -1}},
    {'$limit': 10},
]
t0 = time.time()
via_mongo = pd.DataFrame(list(coleccion.aggregate(TUBERIA)))
t_mongo = time.time() - t0
COLUMNAS = ['naviera', 'buques_en_riesgo', 'factura_ets_eur', 'ahorro_alcanzable_eur']
via_mongo = via_mongo.rename(columns={'_id': 'naviera'})[COLUMNAS]
via_mongo.round(0)

,naviera,buques_en_riesgo,factura_ets_eur,ahorro_alcanzable_eur
0,MSC SHIPMANAGEMENT LIMITED,125,138090309.0,40160518.0
1,MSC Shipmanagement Ltd,105,124421164.0,38001702.0
2,MEDITERRANEAN SHIPPING CO SRL,74,125663276.0,35306014.0
3,Spliethoffs Bevrachtingskantoor B.V.,50,26481389.0,9121998.0
4,CMA CGM SA,43,31529736.0,6135310.0
5,Maersk A/S,32,47824544.0,3934456.0
6,TECHNOMAR SHIPPING INC-LIB,29,22937737.0,7599189.0
7,FLEET MANAGEMENT LIMITED,28,8467742.0,2086710.0
8,Oldendorff Carriers GmbH & Co. KG,28,10994009.0,3769791.0
9,Stolt Tankers BV,26,8529644.0,5602555.0


## 4. ¿Da lo mismo que pandas? Comprobado, no supuesto

Mismo criterio que el notebook 09 con SQL: una tecnología nueva solo entra en el trabajo si se
demuestra que responde **exactamente** lo mismo que el camino ya validado.

In [6]:
t0 = time.time()
riesgo = ficha[ficha.anio_caida.notna() & (ficha.anio_caida <= 2027)]
via_pandas = (riesgo.groupby('company_name')
              .agg(buques_en_riesgo=('ship_imo_number', 'size'),
                   factura_ets_eur=('ets_2026_eur', 'sum'),
                   ahorro_alcanzable_eur=('ahorro_eur', 'sum'))
              .sort_values('buques_en_riesgo', ascending=False).head(10).reset_index())
t_pandas = time.time() - t0
via_pandas = via_pandas.rename(columns={'company_name': 'naviera'})[COLUMNAS]

comparacion = via_mongo.merge(via_pandas, on='naviera', suffixes=('_mongo', '_pandas'))
comparacion['dif_eur'] = (comparacion.factura_ets_eur_mongo
                          - comparacion.factura_ets_eur_pandas).abs()
print('discrepancia maxima en euros:', comparacion.dif_eur.max())
print('mismos buques en riesgo:',
      bool((comparacion.buques_en_riesgo_mongo == comparacion.buques_en_riesgo_pandas).all()))
print(f'tiempo: agregacion {t_mongo*1000:.1f} ms | pandas {t_pandas*1000:.1f} ms '
      f'({t_mongo/t_pandas:.1f}x)')

discrepancia maxima en euros: 5.960464477539063e-08
mismos buques en riesgo: True
tiempo: agregacion 2126.4 ms | pandas 37.3 ms (57.0x)


**Diferencia cero hasta el octavo decimal y el mismo reparto de buques.** La tubería de agregación y
el `groupby` son la misma respuesta.

**Sobre el tiempo hay que ser preciso, porque aquí sí se puede exagerar sin querer.** La agregación
tarda dos órdenes de magnitud más, pero lo que se ha medido **no es MongoDB**: `mongomock` es una
reimplementación en Python puro de la API, sin motor de almacenamiento, sin índices B-tree reales y
sin el ejecutor en C++. Comparar su tiempo con el de pandas no dice nada sobre el coste de un servidor
MongoDB, así que **este notebook no concluye nada sobre rendimiento**: mediría un mock, no la
tecnología. La comparación honesta con el coste medido está en el notebook 09 (SQLite contra pandas,
las dos con motor real) y en el 11 (Spark contra pandas).

Lo que sí se puede afirmar desde aquí es lo del **modelo de datos**, que era la pregunta: la ventaja
documental no está en agregar catorce mil documentos, está en las dos cosas que la tabla ancha no sabe
hacer, y que son las dos consultas siguientes.

## 5. Consulta 3: `$unwind`, o por qué la trayectoria quería ser una lista

*«¿Cuántos buques hay en cada banda, año a año, entre 2025 y 2030?»*

Sobre la tabla ancha hay que apilar seis columnas a mano. Sobre el documento, `$unwind` despliega la
lista y la pregunta se responde sin reestructurar nada. Es la consulta que justifica el modelo.

In [7]:
reparto = pd.DataFrame(list(coleccion.aggregate([
    {'$unwind': '$cii.trayectoria'},
    {'$group': {'_id': {'anio': '$cii.trayectoria.anio', 'banda': '$cii.trayectoria.banda'},
                'n': {'$sum': 1}}},
])))
reparto = pd.DataFrame({'anio': [x['anio'] for x in reparto._id],
                        'banda': [x['banda'] for x in reparto._id],
                        'n': reparto.n})
tabla = (reparto.pivot(index='anio', columns='banda', values='n')
         .reindex(columns=list('ABCDE')).fillna(0).astype(int))
porcentaje = tabla.div(tabla.sum(axis=1), axis=0).mul(100).round(1)
print('Reparto de bandas CII, en % de la flota calificada:')
porcentaje

Reparto de bandas CII, en % de la flota calificada:


banda,A,B,C,D,E
anio,,,,,
2025,17.3,20.2,25.9,19.1,17.6
2026,14.1,19.0,25.9,20.6,20.4
2027,10.8,16.8,25.7,22.2,24.5
2028,7.7,14.3,25.7,22.9,29.4
2029,5.5,11.4,23.9,24.1,35.1
2030,3.8,8.6,21.7,24.8,41.1


**El 36,7% en D/E de 2025 pasa al 65,9% en 2030 sin que ningún buque cambie de comportamiento**, y la
banda A se vacía del 17,3% al 3,8%. Es la trayectoria de la capa regulatoria sobre los buques que
llegan al pasaporte, y aquí sale de una sola consulta sobre el modelo de datos que el producto ya
usaba.

Esta cifra y la que publica la capa regulatoria describen la misma población, porque los dos
scripts aplican el mismo filtro —los buques que la regla 28 del anexo VI de MARPOL cubre de
verdad—, y por eso coinciden. El reparto de 2025 sale 36,7% en D/E aquí y 37,1% en la capa
regulatoria; lo poco que queda son los buques calificables del registro que no tienen predicción
del modelo y por tanto no llegan al pasaporte.

## 6. Consulta 4: lo que la tabla no podía preguntar

El esquema flexible permite **guardar documentos que no tienen la misma forma**, y aquí eso no es una
comodidad: es información. Un buque sin banda CII no es un buque con la banda vacía —es un buque que
el reglamento no califica, y el motivo importa—. En el documento, el campo sencillamente no está, y
`$exists` lo distingue de un nulo. En la tabla ancha las dos cosas son la misma celda vacía.

El pasaporte guarda además **por qué** un buque no se califica, así que la consulta no solo separa
las dos situaciones: dice cuál es cada una. De los 14.865
pasaportes, 9.900 llevan banda y 4.965 no la llevan porque el CII no les alcanza.

In [8]:
sin_banda = coleccion.count_documents({'cii.banda_actual': None})
con_banda = coleccion.count_documents({'cii.banda_actual': {'$ne': None}})
sin_naviera = coleccion.count_documents({'naviera.imo': None})
print(f'con banda CII: {con_banda:,}  |  sin banda: {sin_banda:,}'.replace(',', '.'))
print(f'sin IMO de compania identificado: {sin_naviera:,}'.replace(',', '.'))

# Los que no cumplen FuelEU y ademas caen a D/E antes de 2030: doble frente regulatorio abierto.
doble = list(coleccion.aggregate([
    {'$match': {'fueleu.cumple': False, 'cii.anio_caida': {'$ne': None, '$lte': 2029}}},
    {'$group': {'_id': None, 'buques': {'$sum': 1},
                'factura_eur': {'$sum': '$ets.factura_2026_eur'},
                'multa_fueleu_eur': {'$sum': '$fueleu.penalizacion_eur'}}},
]))[0]
print(f"\nDoble frente abierto: {doble['buques']:,} buques incumplen FuelEU hoy Y pierden la C "
      f"antes de 2030.".replace(',', '.'))
print('   pagan {:,.0f} M EUR de derechos y {:,.0f} M EUR de multa FuelEU al ano'.format(
    doble['factura_eur'] / 1e6, doble['multa_fueleu_eur'] / 1e6).replace(',', '.'))

con banda CII: 9.900  |  sin banda: 4.965
sin IMO de compania identificado: 8



Doble frente abierto: 5.826 buques incumplen FuelEU hoy Y pierden la C antes de 2030.
   pagan 2.727 M EUR de derechos y 519 M EUR de multa FuelEU al ano


## 7. Despliegue: la colección, en JSONL para `mongoimport`

`mongomock` sirve para ejecutar este notebook en cualquier máquina sin levantar un servidor, pero el
artefacto que se entrega es la colección. Se exporta en JSON por líneas, que es lo que come
`mongoimport` sin transformación ninguna:

```bash
mongoimport --db tfm_mrv --collection pasaportes --file data/processed/pasaportes.jsonl
```

In [9]:
os.makedirs(os.path.dirname(RUTA_JSONL), exist_ok=True)
with open(RUTA_JSONL, 'w', encoding='utf-8') as fichero:
    for d in documentos:
        fichero.write(json.dumps(d, ensure_ascii=False) + '\n')
tam = os.path.getsize(RUTA_JSONL) / 1e6
print(f'{RUTA_JSONL}: {len(documentos):,} documentos, {tam:.1f} MB'.replace(',', '.'))

../data/processed/pasaportes.jsonl: 14.865 documentos. 13.7 MB


## 8. Conclusiones

**1. El modelo documental no sustituye al relacional aquí: sirve a otra capa.** El registro MRV es
tabular y el notebook 09 demuestra que SQL le saca cosas que pandas no había visto. El *producto* —el
pasaporte de cumplimiento— es un objeto anidado con una lista de longitud variable, y ahí el documento
es el modelo natural. Elegir según el dato, y no según la moda, es lo importante de la decisión.

**2. Y una medición que este notebook se niega a hacer.** La agregación tarda mucho más que el
`groupby`, pero lo ejecutado es `mongomock` —Python puro, sin motor— y no MongoDB, así que ese número
no mide la tecnología y no se publica como si lo hiciera. Decirlo cuesta un párrafo y evita una
conclusión sin fundamento. Las comparaciones de coste con motor real están donde
corresponde: SQLite en el notebook 09 y Spark en el 11.

**3. Lo que sí gana es la forma de la pregunta.** `$unwind` sobre la trayectoria responde en una línea
lo que sobre la tabla ancha exige apilar seis columnas, y `$exists` distingue *«no calificable»* de
*«sin dato»*, que en la tabla son la misma celda vacía y en el reglamento son cosas distintas.

**4. Un resultado que el modelo destapa**: hay buques con los dos frentes regulatorios abiertos a la
vez —incumplen FuelEU hoy **y** pierden la C antes de 2030—. Ninguna de las dos capas los veía por
separado, porque cada una miraba su propio reglamento.